# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Divyanshu10045/flyrank-assignments/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 model the way I would want it audited, and then rewrites my claims so they match what the numbers can actually carry. It rebuilds the exact ML-04 / ML-07 / ML-08 frame and label, trains the same from-scratch model, and changes only how the evaluation is split and how the features are attributed.

**What each section does**

1. Two findings from `docs/flyrank-seo-research-march-2026.pdf`, and the methodology question each one raises about my own notebook.
2. The Week-5 model before and after: the split ML-08 actually shipped, then a client-grouped out-of-fold evaluation with the penalty chosen inside the training folds only.
3. The Week-3 leakage hunt, re-run on the final feature set, plus the one frame-construction step whose outcome-conditioned membership I measure rather than assume is a problem.
4. My boldest Week-5 sentence, rewritten in safe language, with the measured number behind each rewrite.

**Continuity (non-negotiable)**

- Frame: features 2026-01-01 -> 2026-02-28 (59 days), label window 2026-03-01 -> 2026-03-31 (31 days), decision point 2026-03-01.
- Counts asserted by count, not eyeballed: pool 67,279 rows / 7,445 positives / 24 clients, asserted exactly against the ML-08 receipt's base rate. The frame comes out 8 rows and 1 client lighter than ML-04 recorded (119,115 / 28 against 119,123 / 29) — that drift sits entirely outside the pool, the cell prints both figures, and Section 3 says what I did and did not do about it.
- Contract features stay exactly the five ML-04 features. `content_updated_date` stays excluded. `trend_pct_win` is carried only as ML-08's probe model and is audited like any other feature.
- No April or later partition is read. No `_sample` partition is read. No DuckDB. The compiled ML libraries are blocked on this box by Application Control, so the fitter stays the numpy/scipy one from ML-08.
- One environment change since ML-08: `pyarrow.parquet` is now blocked too ("An Application Control policy has blocked this file" on `_parquet.pyd`), so the parquet reads go through **fastparquet** instead. Same files, same columns, same filters, same aggregates — only the reader changed. If you re-run this on a box where pyarrow loads, nothing else differs.
- This notebook builds its own monthly cache (`ml09_cache`) instead of reusing the shared `ml04_cache`, because those aggregates are at a coarser grain with null `content_hash_id` rows and cannot be joined one-to-one. The pool it produces is byte-identical to ML-04/ML-07/ML-08, which is asserted below.

> **DRAFT** — every section below is a working draft until the self-check at the end is ticked.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**DRAFT**

I read the March report looking for things I could check against my own notebook rather than things I could quote. Two exhibits earned a place here, because both make a habit visible: each one states where its label comes from before it states what the label predicts, and both are honest about which claims the design cannot carry. That is the standard I want to hold my own Week-5 result to, so I wrote each finding down as a question aimed at me.

**Finding A — "The Anatomy of Growing Content" (Finding #1, CONFIRMED, p. 6).** The report compares pages with rising impressions against pages with falling impressions and reports that growing pages are longer and younger: 74,187 rising pages at 3.2K average words and 184 days average age, against 45,272 declining pages at 2.3K words and 230 days, which the report rounds to "37.6% longer … and 20% younger."

Where the label comes from is stated on p. 5: `trend_direction` is "Calculated from 30d-vs-prev-30d impression change. Up: >10% growth. Down: >10% decline." So the up/down label and the profiling columns — words, age, position, health — are all measured inside the same 30-day window. The report does not hide this. It calls the result "an observational comparison" on p. 6 and scopes the whole study as observational again on p. 36: "correlations do not prove causation."

Does the validation design carry the claim? It carries the profile claim honestly, because the cohorts are large and the comparison is a plain aggregate contrast. It does not carry the recommendation attached to it. The "What to Do With This" step on p. 6 says "Expand thin pages that already earn impressions," with an expected outcome that "an already visible page can keep growing." A same-window contrast between two cohorts cannot show that adding words causes retention; it can only show that pages which grew were, on average, already longer. The report's own evidence standard on p. 4 is the reason this is a question rather than a complaint — headline findings "prioritize direct aggregate comparisons," and a descriptive aggregate is exactly what was run.

My three questions, aimed at my own work:

1. Would the up-versus-down gap survive if "growing" meant *next*-period growth rather than growth inside the window being profiled? This is the one I most wanted to test, because it decides whether my label is measuring a future outcome or re-measuring the present.
2. The up-versus-down split is never printed as a share of the portfolio, so a reader cannot tell how lopsided the two cohorts were before trusting the two averages.
3. Average position in that same table is 15.9 versus 16.1 — essentially a tie — yet the narrative lists "already positioned slightly better" as one of the three differences. Is a difference that small worth carrying in the sentence?

**Finding B — "What Predicts Growth?" (ML appendix, p. 29).** A logistic regression is reported at "71% holdout accuracy describing which sampled features separate growing from declining pages," with content age, days since update and days visible as the strongest coefficients and impressions, clicks and sessions at zero.

Where the label comes from is the same `trend_direction` bucket from Finding A, so the label and the feature window overlap in exactly the same way. The split is described on p. 36 as one of three models run at "(80/20 split)," over a sample of 61.8K content pieces restricted to "sessions > 0 and impressions > 0."

Does the validation design carry the claim? The report scopes it as exploratory in three separate places — p. 4 ("ML pages are exploratory appendix material and do not override direct portfolio evidence"), p. 5 (the ML techniques "appear only in the exploratory appendix"), and p. 36 ("ML pages remain exploratory and secondary to direct aggregate comparisons") — and p. 36 adds that "No p-values or confidence intervals are reported." So the exhibit is not overclaiming, and I want to be careful not to read it as overclaiming. What it leaves unprinted is what I actually needed.

My three questions:

1. What does a majority-class rule score on that same 20%? Accuracy on a two-class label is unreadable without the base rate, and the sample is pre-filtered to pages with sessions and impressions, which is itself a selection that moves the base rate.
2. Does the ranking survive holding out whole brands? The portfolio has 57 brands and the split is described only as 80/20, so pages from the same brand can sit on both sides and the model can lean on brand-level noise it will not have at scoring time.
3. The label is a change in impressions while impressions is also listed among the candidate features. Does that overlap need to be stated outright?

**How I answer them in my own lane.** The cell below answers question A1 with an assertion instead of an argument: it traces every feature I use to a source month and proves nothing dated after the decision point reaches a feature, and that my label window opens the day after my feature window closes. A2 and B1 are answered by printing the base rate next to every ranking number, because that is the number the paper leaves out and the one I most often leave out too. A3 I answer by not making the claim — my Section 4 rewrite drops the position sentence rather than softening it.

I want to flag what I cannot answer. Finding A's question 1 asks whether a profile survives a *next-period* definition of growth, and the honest test needs a second forward window after March. The lane is frozen at decision point 2026-03-01 and I am not reading April, so I can prove my label is forward-looking relative to my features but I cannot prove my result would survive another period. That is a limitation of my evidence, not a result, and Section 4 says so.

Finding B's question 2 I can answer directly, and Section 2 does: my before/after is a client-grouped out-of-fold evaluation, which is the brand-level version of that question.

In [1]:
# --- setup: rebuild the exact ML-04 / ML-07 / ML-08 frame and label ---
# Reader note: pyarrow's Parquet DLL is blocked by Application Control on this box
# (pyarrow.compute and pyarrow.csv still import). fastparquet reads the same files,
# so the only thing that changed since ML-08 is the reader.
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import fastparquet
from huggingface_hub import HfFileSystem
from dotenv import load_dotenv
from scipy.optimize import minimize

REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"work"/"notebooks").is_dir() and (p/".git").exists()), Path.cwd())
os.chdir(REPO)
load_dotenv(REPO/".env")
fs = HfFileSystem(token=os.environ.get("HF_TOKEN") or None)

WAREHOUSE = "datasets/FlyRank/internship-warehouse"
FACT = WAREHOUSE + "/fact_content_daily_performance"
KEYS = ["client_hash_id", "content_hash_id"]

FEATURE_START, FEATURE_END = "2026-01-01", "2026-02-28"
LABEL_START,   LABEL_END   = "2026-03-01", "2026-03-31"
DECISION_POINT = LABEL_START
PRIOR_DAYS, LABEL_DAYS = 59, 31
DAYS_IN = {"2026-01": 31, "2026-02": 28, "2026-03": 31}
POOL_FLOOR, VOLUME_FLOOR, DROP_FLOOR = 200, 500, -20
FEATURE_MONTHS, LABEL_MONTHS = ("2026-01", "2026-02"), ("2026-03",)
DAILY_COLS = ["report_date", "client_hash_id", "content_hash_id",
              "gsc_impressions", "gsc_clicks", "gsc_sum_position", "gsc_data_available"]
ENGINE = "fastparquet"

# NB this notebook does NOT reuse the shared .../opencode/ml04_cache directory. Those aggregates sit
# at a different grain: they carry ~84k rows per month with a null content_hash_id, which makes the
# joins below many-to-many and the frame unreproducible. ml09_cache holds one row per
# (client_hash_id, content_hash_id), which is the unit of analysis ML-04 describes.
CACHE = Path(os.environ.get("TEMP", "."))/"opencode"/"ml09_cache"
CACHE.mkdir(parents=True, exist_ok=True)

def read_parquet(path, columns=None):
    with fs.open(path) as fh:
        return fastparquet.ParquetFile(fh).to_pandas(columns=columns)

def month_file(m):
    parts = sorted(p for p in fs.ls(FACT+"/month="+m, detail=False) if p.endswith(".parquet"))
    if not parts:
        raise FileNotFoundError(m)
    return parts[0]

def agg_month(m, my_clients):
    """monthly totals per (client, content). NB the positive-impression day filter:
    it decides who is even present in a month, and Section 3 measures what that costs."""
    cached = CACHE/("agg_%s.parquet" % m)
    if cached.exists():
        g = pd.read_parquet(cached, engine=ENGINE)
        return g[g["client_hash_id"].isin(my_clients)].copy()
    t = read_parquet(month_file(m), DAILY_COLS)
    if my_clients:
        t = t[t["client_hash_id"].isin(my_clients)]
    t = t[t["gsc_data_available"].fillna(False).astype(bool)]
    t = t[t["gsc_impressions"] > 0]
    g = (t.groupby(KEYS, as_index=False)
           .agg(active_days=("report_date", "count"), impr=("gsc_impressions", "sum"),
                clicks=("gsc_clicks", "sum"), sum_pos=("gsc_sum_position", "sum")))
    # a unit of analysis is one (client, content) pair, so a row with no content id is not a unit.
    # Guarding this also keeps the joins below one-to-one instead of many-to-many.
    assert g[KEYS].notna().all().all(), ("null join key in", m)
    assert not g.duplicated(KEYS).any(), ("duplicate join key in", m)
    g["month"] = m
    g.to_parquet(cached, engine=ENGINE)
    return g

clients = read_parquet(WAREHOUSE+"/dim_clients.parquet",
                       ["client_hash_id", "gsc_data_start", "has_gsc_access"])
gsc_start = pd.to_datetime(clients["gsc_data_start"].astype("string"))
MY_CLIENTS = set(clients.loc[(gsc_start <= FEATURE_START) & (clients["has_gsc_access"] != False),
                             "client_hash_id"])
monthly = {m: agg_month(m, MY_CLIENTS) for m in FEATURE_MONTHS + LABEL_MONTHS}

j  = monthly["2026-01"].rename(columns={"impr": "impr_jan", "clicks": "clicks_jan",
                                        "sum_pos": "sum_pos_jan", "active_days": "days_jan"})
b  = monthly["2026-02"].rename(columns={"impr": "impr_feb", "clicks": "clicks_feb",
                                        "sum_pos": "sum_pos_feb", "active_days": "days_feb"})
lb = monthly["2026-03"][KEYS+["impr"]].rename(columns={"impr": "impr_label"})

# the March join is an INNER join on purpose -- Section 3 measures what that costs
f = j.merge(b, on=KEYS, how="outer").merge(lb, on=KEYS, how="inner")

f["impr_prior"] = f["impr_jan"].fillna(0) + f["impr_feb"].fillna(0)
f["clicks_prior"] = f["clicks_jan"].fillna(0) + f["clicks_feb"].fillna(0)
f["sum_pos_prior"] = f["sum_pos_jan"].fillna(0) + f["sum_pos_feb"].fillna(0)
f["active_days_prior"] = f["days_jan"].fillna(0) + f["days_feb"].fillna(0)
f["ctr_prior_pct"] = np.where(f["impr_prior"] > 0, f["clicks_prior"]/f["impr_prior"]*100, np.nan)
f["wtd_pos_prior"] = np.where(f["impr_prior"] > 0, f["sum_pos_prior"]/f["impr_prior"], np.nan)
f["jan_rate_30d"] = f["impr_jan"].fillna(0)/DAYS_IN["2026-01"]*30
f["feb_rate_30d"] = f["impr_feb"].fillna(0)/DAYS_IN["2026-02"]*30
f["trend_pct_win"] = np.where(f["jan_rate_30d"] > 0,
                              (f["feb_rate_30d"]-f["jan_rate_30d"])/f["jan_rate_30d"]*100, np.nan)

content = read_parquet(WAREHOUSE+"/dim_content.parquet",
                       ["content_hash_id", "content_created_date",
                        "content_updated_date", "word_count"])
for c in ("content_created_date", "content_updated_date"):
    content[c] = pd.to_datetime(content[c].astype("string"))
f = f.merge(content, on="content_hash_id", how="left")
f["content_age_days"] = (pd.Timestamp(DECISION_POINT) - f["content_created_date"]).dt.days

f["prior_rate_30d"] = f["impr_prior"]/PRIOR_DAYS*30
f["label_rate_30d"] = f["impr_label"]/LABEL_DAYS*30
f["drop_pct"] = np.where(f["prior_rate_30d"] > 0,
                         (f["label_rate_30d"] - f["prior_rate_30d"])/f["prior_rate_30d"]*100, np.nan)
f["in_pool"] = f["impr_prior"] >= POOL_FLOOR
f["is_material_decline"] = (f["in_pool"] & (f["prior_rate_30d"] >= VOLUME_FLOOR)
                            & (f["drop_pct"] <= DROP_FLOOR)).astype(int)

P = f[f["in_pool"]].copy()
P["log_impr_prior"] = np.log1p(P["impr_prior"])
P["log_impr_label_leak"] = np.log1p(P["impr_label"])     # Section 3 trap only, never contracted

# The pool is asserted exactly, because every number in this notebook is computed on it.
assert int(f["in_pool"].sum()) == 67279, int(f["in_pool"].sum())
assert int(f.loc[f["in_pool"], "is_material_decline"].sum()) == 7445
assert P["client_hash_id"].nunique() == 24
assert float(P["is_material_decline"].mean()) == 0.11065860075209204, float(P["is_material_decline"].mean())
# The frame is wider than the pool, and it differs slightly from the frame ML-04 recorded: 8 fewer
# rows and one fewer client. Every difference sits OUTSIDE the pool, so no number in this notebook
# moves. Asserted as a tolerance rather than an equality, and printed either way, because the
# honest thing is to show the drift instead of pinning a stale figure.
FRAME_ROWS_ML04, FRAME_CLIENTS_ML04 = 119123, 29
frame_drift = len(f) - FRAME_ROWS_ML04
assert abs(frame_drift) <= 0.001*FRAME_ROWS_ML04, ("frame drifted", len(f))

FEATS5 = ["log_impr_prior", "ctr_prior_pct", "wtd_pos_prior", "active_days_prior", "content_age_days"]
FEATS6 = FEATS5 + ["trend_pct_win"]
LEAK = "log_impr_label_leak"
y = P["is_material_decline"].to_numpy()
cid = P["client_hash_id"].to_numpy()
BASE = float(y.mean())
print("pool %s rows / %d clients | positives %s | base %.4f   <- exact, asserted"
      % ("{:,}".format(len(P)), P["client_hash_id"].nunique(), "{:,}".format(int(y.sum())), BASE))
print("frame %s rows / %d clients   <- ML-04 recorded %s / %d, drift %+d rows (%.3f%%), pool unaffected"
      % ("{:,}".format(len(f)), f["client_hash_id"].nunique(), "{:,}".format(FRAME_ROWS_ML04),
         FRAME_CLIENTS_ML04, frame_drift, 100.0*frame_drift/FRAME_ROWS_ML04))

# --- finding A: is my label in a different window from my features? ---
# Provenance is declared once and asserted, not described in prose and hoped for.
PROVENANCE = [
    dict(col="log_impr_prior",      src="2026-01 + 2026-02",        latest="2026-02-28", perf=True,  role="feature"),
    dict(col="ctr_prior_pct",       src="2026-01 + 2026-02",        latest="2026-02-28", perf=True,  role="feature"),
    dict(col="wtd_pos_prior",       src="2026-01 + 2026-02",        latest="2026-02-28", perf=True,  role="feature"),
    dict(col="active_days_prior",   src="2026-01 + 2026-02",        latest="2026-02-28", perf=True,  role="feature"),
    dict(col="content_age_days",    src="dim_content.created_date", latest="2026-03-01", perf=False, role="feature"),
    dict(col="trend_pct_win",       src="2026-01 + 2026-02",        latest="2026-02-28", perf=True,  role="probe feature"),
    dict(col=LEAK,                  src="2026-03",                  latest="2026-03-31", perf=True,  role="leak demo only"),
    dict(col="impr_label",          src="2026-03",                  latest="2026-03-31", perf=True,  role="label only"),
    dict(col="label_rate_30d",      src="2026-03",                  latest="2026-03-31", perf=True,  role="label only"),
    dict(col="drop_pct",            src="2026-03 vs 2026-01+02",     latest="2026-03-31", perf=True,  role="label only"),
]
for r in PROVENANCE:
    if r["role"] in ("feature", "probe feature"):
        assert pd.Timestamp(r["latest"]) <= pd.Timestamp(DECISION_POINT), ("past the decision point", r)
    if r["role"] == "feature" and r["perf"]:
        assert pd.Timestamp(r["latest"]) <= pd.Timestamp(FEATURE_END), ("past the feature window", r)
print()
print("decision point %s | feature window %s -> %s | label window %s -> %s"
      % (DECISION_POINT, FEATURE_START, FEATURE_END, LABEL_START, LABEL_END))
print("provenance rows asserted: %d | features that may read performance data must end by %s"
      % (len(PROVENANCE), FEATURE_END))
print("overlap between the feature window and the label window: %d days (they are adjacent, not shared)"
      % max(0, (pd.Timestamp(FEATURE_END) - pd.Timestamp(LABEL_START)).days + 1))
print("columns that may touch the label window:",
      [r["col"] for r in PROVENANCE if r["role"] in ("label only", "leak demo only")])

# --- finding B: the base rate the paper prints next to no metric ---
M07 = json.load(open(REPO/"work"/"outputs"/"baseline_metrics.json", encoding="utf-8"))
M08 = json.load(open(REPO/"work"/"outputs"/"ml08_metrics.json", encoding="utf-8"))
print()
print("pool base rate %.4f -> a random top-50 queue holds about %.1f positives by chance"
      % (BASE, 50*BASE))
print("ML-07 frozen rule, this pool: P@20 %.3f | P@50 %.3f | P@200 %.3f | random floor %.3f"
      % (M07["p_at_20"], M07["p_at_50"], M07["p_at_200"], M07["random_floor_p_at_50"]))
print("ML-07 forward Feb->Mar probe P@50 %.3f at base %.3f -- the caveat that travels with 0.720"
      % (M07["forward_label_p_at_50"], M07["forward_label_base_rate"]))
print("ML-08 shipped: best_l2 %s | model A val P@50 %.2f | model A full P@50 %.2f"
      % (M08["best_l2"], M08["model_A_val"]["P@50"], M08["model_A_full"]["P@50"]))

C:\Users\dk732\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pool 67,279 rows / 24 clients | positives 7,445 | base 0.1107   <- exact, asserted
frame 119,115 rows / 28 clients   <- ML-04 recorded 119,123 / 29, drift -8 rows (-0.007%), pool unaffected

decision point 2026-03-01 | feature window 2026-01-01 -> 2026-02-28 | label window 2026-03-01 -> 2026-03-31
provenance rows asserted: 10 | features that may read performance data must end by 2026-02-28
overlap between the feature window and the label window: 0 days (they are adjacent, not shared)
columns that may touch the label window: ['log_impr_label_leak', 'impr_label', 'label_rate_30d', 'drop_pct']

pool base rate 0.1107 -> a random top-50 queue holds about 5.5 positives by chance
ML-07 frozen rule, this pool: P@20 0.800 | P@50 0.720 | P@200 0.695 | random floor 0.111
ML-07 forward Feb->Mar probe P@50 0.340 at base 0.207 -- the caveat that travels with 0.720
ML-08 shipped: best_l2 0.0001 | model A val P@50 0.24 | model A full P@50 0.48


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**DRAFT**

The model does not change in this section. Same five contract features, same degree-2 polynomial design, same numpy/scipy fitter, same label, same pool — and the reproduction check below confirms it, hitting ML-08's shipped 0.24 and 0.48 to three decimals. What changes is how the evaluation is carved up, and the number moves for three separable reasons. I want to keep them apart, because two of them push the number down and one pushes it up, and averaging them into a single "before vs after" would hide the one that matters most.

**Cause 1 — the full-pool score was partly in-sample, and it dominated.** ML-08 fitted on 18 clients and then scored the design matrix built over the *entire* pool, including the 65,489 training rows out of 67,279. The result is not a small contamination, it is the whole number: 0.480 in-sample against 0.240 on the six held-out clients, from the same fitted model, with the training rows making up 97.3% of the pool and outnumbering the holdout 37 to 1. That is a training score wearing a different label. The same holds for the probe model, 0.520 in-sample against 0.240 held-out.

**Cause 2 — the penalty was chosen on the holdout it then reported.** ML-08 picked its L2 by maximising Precision@50 on the six validation clients and then reported Precision@50 on those same clients. Re-selecting the penalty by inner grouped cross-validation on the training clients only moves the held-out number from 0.240 to 0.220, so the direct cost of that habit is small here — about 0.02 P@50. But the more interesting part is *which* penalty gets chosen: tuning against the holdout picks 1e-4, the weakest regularisation on the grid, while honest inner selection picks 1.0 in every one of the six folds, the strongest. Both choices are defensible models; only one of them was picked without peeking, and I would rather ship the second.

**Cause 3 — and this is the one I did not expect — one draw of six clients was carrying the claim, and it was carrying it downwards.** The seed-42 holdout has a base rate of 0.0782 against the pool's 0.1107: it is the easier slice, and the model was being graded on it. Replacing it with six-fold client-grouped out-of-fold prediction, where every pool row is scored by a model that never saw it and the whole 24-client pool is covered, *raises* Precision@50 from 0.240 to 0.300. The honest split is not a punishment here. It is still the right number, because the per-fold spread runs from 0.260 to 0.540 with a standard deviation of 0.098 across only four clients per fold — a single draw could have landed anywhere in that band, and the one ML-08 happened to use happened to be a soft one.

So the before/after is not one arrow, it is three: the full-pool 0.480 is 2.0x the held-out 0.240 on the same fit purely because 97.3% of its rows are training rows; 0.240 was pessimistic because of which six clients got held out; and 0.300 is the figure I would put my name against — 1.6x below the full-pool number, and carrying a client-level spread I am reporting rather than hiding.

One result inverts Week 5. Model B adds the Jan-to-Feb slope and was reported in ML-08 as the reason the baseline's 0.720 looked reachable. Out-of-fold, the probe is **worse** than the contract model — 0.280 against 0.300 — and Section 4 shows why: scrambling the slope costs 0.0007 Precision@50, which is nothing. Both sit about 2.7x and 2.5x above the 0.1107 random floor, so the contract model is doing real work, but the slope is not what it is doing.

I am not claiming the model beats ML-07. ML-07's 0.720 is on the same blended label, and its own forward February-to-March probe was 0.340, so 0.720 and 0.300 are not measuring the same thing. What I can say is narrower and more useful: on one forward window, under a split where no client appears on both sides, the five contracted features put roughly 2.7x the base rate in the top 50, and the slope adds nothing I can detect.

In [2]:
# --- the ML-08 fitter, unchanged: degree-2 logistic regression, L2, numpy/scipy ---
def build_design(A_raw, med, mu, sd, miss_cols):
    A = A_raw.astype(float).copy()
    for jj in miss_cols:
        col = A[:, jj]
        col[np.isnan(col)] = med[jj]
    Z = (A - mu)/sd
    parts = [Z, Z**2]
    n = Z.shape[1]
    for i in range(n):
        for k in range(i+1, n):
            parts.append((Z[:, i]*Z[:, k]).reshape(-1, 1))
    for jj in miss_cols:
        parts.append(np.isnan(A_raw[:, jj]).astype(float).reshape(-1, 1))
    Z2 = np.hstack(parts)
    return np.hstack([np.ones((Z2.shape[0], 1)), Z2])

def _sig(z):
    return 1.0/(1.0 + np.exp(-np.clip(z, -30, 30)))

def train_logreg(Z, yy, l2):
    yy = yy.astype(float)
    n = len(yy); npos = yy.sum(); nneg = n - npos
    sw = np.where(yy == 1, n/(2.0*npos), n/(2.0*nneg))
    def obj(w):
        p = _sig(Z@w); eps = 1e-12
        nll = -np.sum(sw*(yy*np.log(p+eps) + (1-yy)*np.log(1-p+eps)))/n
        return nll + 0.5*l2*np.sum(w[1:]**2)
    def grad(w):
        p = _sig(Z@w)
        g = Z.T@(sw*(p-yy))/n
        g[1:] += l2*w[1:]
        return g
    res = minimize(obj, np.zeros(Z.shape[1]), jac=grad, method="L-BFGS-B", options={"maxiter": 500})
    return res.x, float(res.fun)

def prec_at_k(scores, labels, k):
    s = np.asarray(scores); l = np.asarray(labels)
    return float(l[np.argsort(-s, kind="stable")[:k]].mean())

def pak(scores, labels):
    l = np.asarray(labels)
    return {"P@20": prec_at_k(scores, l, 20), "P@50": prec_at_k(scores, l, 50),
            "P@200": prec_at_k(scores, l, 200), "n": int(len(l)), "pos": int(l.sum()),
            "base": float(l.mean())}

GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]
XD = {}
def mat(featnames):
    key = tuple(featnames)
    if key not in XD:
        XD[key] = P[list(featnames)].to_numpy(float)     # list(): pandas 3 reads a tuple as one key
    return XD[key]

def fit_on(idx, featnames, l2):
    """standardise and expand on the rows handed in, then fit. Nothing else is touched."""
    D = mat(featnames)[idx]
    med = np.nanmedian(D, axis=0)
    miss = np.where(np.isnan(D).any(axis=0))[0]
    mu = np.nanmean(D, axis=0); sd = np.nanstd(D, axis=0); sd[sd == 0] = 1.0
    w, nll = train_logreg(build_design(D, med, mu, sd, miss), y[idx], l2)
    return dict(w=w, med=med, mu=mu, sd=sd, miss=miss, nll=nll)

def score(model, idx, featnames):
    return build_design(mat(featnames)[idx], model["med"], model["mu"], model["sd"], model["miss"]) @ model["w"]

def pick_l2(idx, featnames, n_inner=3, seed=7):
    """choose the penalty from an inner client-grouped split of idx only.
    The rows we are going to report on are never in idx, so they cannot leak in here."""
    c_of = cid[idx]
    cl = np.array(sorted(set(c_of)))
    perm = np.random.default_rng(seed).permutation(len(cl))
    grid = {}
    for l2 in GRID:
        vals = []
        for k in range(n_inner):
            hold = set(cl[perm[k::n_inner]])
            hm = np.array([c in hold for c in c_of])
            it, ih = idx[~hm], idx[hm]
            if len(it) == 0 or len(ih) == 0:
                continue
            vals.append(prec_at_k(score(fit_on(it, featnames, l2), ih, featnames), y[ih], 50))
        if vals:
            grid[l2] = float(np.mean(vals))
    return max(grid, key=grid.get), grid

MODELS = [("A contract (5 feats)", tuple(FEATS5)), ("B probe (+trend_pct_win)", tuple(FEATS6))]
ALL = np.arange(len(P))

# =============================== BEFORE: the split ML-08 shipped ===============================
SEED = 42
clients_all = np.array(sorted(set(cid)))
perm = np.random.default_rng(SEED).permutation(len(clients_all))
n_val = max(1, int(round(0.25*len(clients_all))))
c_val = set(clients_all[perm[:n_val]]); c_train = set(clients_all[perm[n_val:]])
mask_tr = np.array([c in c_train for c in cid]); mask_v = np.array([c in c_val for c in cid])
idx_tr, idx_v = np.where(mask_tr)[0], np.where(mask_v)[0]
assert not (c_train & c_val), "a client appears on both sides"
print("BEFORE split (client-grouped, seed %d): train %d clients / %s rows / %s pos | val %d clients / %s rows / %s pos"
      % (SEED, len(c_train), "{:,}".format(len(idx_tr)), "{:,}".format(int(y[idx_tr].sum())),
         len(c_val), "{:,}".format(len(idx_v)), "{:,}".format(int(y[idx_v].sum()))))
print("val base rate %.4f vs pool base rate %.4f -- the holdout is the easier slice"
      % (y[idx_v].mean(), BASE))
print()

BEFORE = {}
for name, feats in MODELS:
    # ML-08 shipped this: penalty chosen by maximising P@50 on the clients it then reported on
    tuned = {}
    for l2 in GRID:
        m = fit_on(idx_tr, feats, l2)
        tuned[l2] = prec_at_k(score(m, idx_v, feats), y[idx_v], 50)
    l2_ship = max(tuned, key=tuned.get)
    m_ship = fit_on(idx_tr, feats, l2_ship)
    row = dict(feats=list(feats), l2_shipped=l2_ship, shipped_grid=tuned,
               train_in_sample=pak(score(m_ship, idx_tr, feats), y[idx_tr]),
               val_reported=pak(score(m_ship, idx_v, feats), y[idx_v]),
               full_pool_mixed=pak(score(m_ship, ALL, feats), y))
    # isolate cause 2: same split, penalty chosen without touching the holdout
    l2_nest, nest_grid = pick_l2(idx_tr, feats, seed=17)
    row["l2_nested"] = l2_nest
    row["nested_grid"] = nest_grid
    row["val_nested_l2"] = pak(score(fit_on(idx_tr, feats, l2_nest), idx_v, feats), y[idx_v])
    BEFORE[name] = row

    print("=" * 96)
    print(name)
    print("  penalty grid scored on the val clients (what ML-08 optimised):",
          {k: round(v, 4) for k, v in tuned.items()}, "-> chose", l2_ship)
    print("  %-34s %8s %8s %8s   %10s %8s" % ("slice", "P@20", "P@50", "P@200", "base", "rows"))
    for lbl, key in (("train rows (IN-SAMPLE)", "train_in_sample"),
                     ("held-out clients (reported)", "val_reported"),
                     ("full pool (mixed, ML-08's \"full\")", "full_pool_mixed"),
                     ("held-out clients, nested L2", "val_nested_l2")):
        r = row[key]
        print("  %-34s %8.3f %8.3f %8.3f   %10.4f %8s"
              % (lbl, r["P@20"], r["P@50"], r["P@200"], r["base"], "{:,}".format(r["n"])))
    print("  cause 2 (tuned on its own holdout): val P@50 %.3f -> %.3f with nested L2 (chose %s)"
          % (row["val_reported"]["P@50"], row["val_nested_l2"]["P@50"], l2_nest))
    print("  cause 1 (in-sample rows in the \"full\" score): %.3f in-sample vs %.3f held-out"
          % (row["train_in_sample"]["P@50"], row["val_reported"]["P@50"]))

print()
print("=" * 96)
print("cross-check against work/outputs/ml08_metrics.json (same fit, must reproduce)")
for name, keyA, keyB in (("A contract (5 feats)", "model_A_val", "model_A_full"),
                         ("B probe (+trend_pct_win)", "model_B_val", "model_B_full")):
    print("  %-24s shipped val P@50 %.2f vs here %.3f | shipped full P@50 %.2f vs here %.3f"
          % (name, M08[keyA]["P@50"], BEFORE[name]["val_reported"]["P@50"],
             M08[keyB]["P@50"], BEFORE[name]["full_pool_mixed"]["P@50"]))

# =============================== AFTER: client-grouped out-of-fold ===============================
K, N_INNER = 6, 3
def run_oof(featnames, k_folds=K, seed=SEED, n_inner=N_INNER):
    oof = np.full(len(P), np.nan)
    recs, held = [], []
    cl = np.array(sorted(set(cid)))
    pk = np.random.default_rng(seed).permutation(len(cl))
    for k in range(k_folds):
        hold = set(cl[pk[k::k_folds]])
        hm = np.array([c in hold for c in cid])
        itr, iva = np.where(~hm)[0], np.where(hm)[0]
        l2, _ = pick_l2(itr, featnames, n_inner=n_inner, seed=100+k)
        m = fit_on(itr, featnames, l2)
        s = score(m, iva, featnames)
        oof[iva] = s
        recs.append(dict(fold=k+1, hold_clients=len(hold), n=len(iva), pos=int(y[iva].sum()),
                         base=float(y[iva].mean()), l2=l2, p50=prec_at_k(s, y[iva], 50)))
        held.append((m, iva))
    assert not np.isnan(oof).any(), "every pool row must be scored exactly once"
    return oof, recs, held

AFTER = {}
for name, feats in MODELS:
    oof, recs, held = run_oof(feats)
    AFTER[name] = dict(oof=pak(oof, y), folds=recs, scores=oof, held=held)
    print()
    print("=" * 96)
    print("AFTER -- %d-fold client-grouped out-of-fold, penalty chosen inside each fold's train clients (%s)"
          % (K, name))
    print("  %-6s %14s %8s %9s %9s %12s" % ("fold", "hold clients", "rows", "pos", "base", "P@50"))
    for r in recs:
        print("  %-6d %14d %8s %9s %9.4f %12.4f"
              % (r["fold"], r["hold_clients"], "{:,}".format(r["n"]), "{:,}".format(r["pos"]),
                 r["base"], r["p50"]))
    p50s = [r["p50"] for r in recs]
    o = AFTER[name]["oof"]
    print("  pooled out-of-fold: P@20 %.3f | P@50 %.3f | P@200 %.3f | base %.4f | rows %s | pos %s"
          % (o["P@20"], o["P@50"], o["P@200"], o["base"], "{:,}".format(o["n"]), "{:,}".format(o["pos"])))
    print("  per-fold P@50 spread: min %.3f | max %.3f | sd %.3f  -> cause 3: one draw was carrying the claim"
          % (min(p50s), max(p50s), float(np.std(p50s, ddof=1))))
    print("  penalty chosen per fold:", [r["l2"] for r in recs])

print()
print("=" * 96)
print("BEFORE / AFTER, P@50 against the base rate (floor = %.4f, ML-07 rule = %.3f)" % (BASE, M07["p_at_50"]))
print("  %-24s %26s %26s %26s" % ("model", "BEFORE full pool (mixed)", "BEFORE val (L2 tuned on val)", "AFTER out-of-fold"))
for name, _ in MODELS:
    bef, aft = BEFORE[name], AFTER[name]["oof"]
    print("  %-24s %26.3f %26.3f %26.3f"
          % (name, bef["full_pool_mixed"]["P@50"], bef["val_reported"]["P@50"], aft["P@50"]))
    print("  %-24s %26.3f %26.3f %26.3f   <- row 1 = the headline three, row 2 = in-sample / nested / OOF"
          % ("", bef["train_in_sample"]["P@50"], bef["val_nested_l2"]["P@50"], aft["P@50"]))
print()
print("  lift over the random floor, AFTER out-of-fold:")
for name, _ in MODELS:
    print("    %-24s %.3f / %.4f = %.2fx" % (name, AFTER[name]["oof"]["P@50"], BASE,
                                            AFTER[name]["oof"]["P@50"]/BASE))

BEFORE split (client-grouped, seed 42): train 18 clients / 65,489 rows / 7,305 pos | val 6 clients / 1,790 rows / 140 pos
val base rate 0.0782 vs pool base rate 0.1107 -- the holdout is the easier slice



A contract (5 feats)
  penalty grid scored on the val clients (what ML-08 optimised): {0.0001: 0.24, 0.001: 0.24, 0.01: 0.24, 0.1: 0.22, 1.0: 0.22} -> chose 0.0001
  slice                                  P@20     P@50    P@200         base     rows
  train rows (IN-SAMPLE)                0.500    0.480    0.435       0.1115   65,489
  held-out clients (reported)           0.300    0.240    0.265       0.0782    1,790
  full pool (mixed, ML-08's "full")     0.500    0.480    0.435       0.1107   67,279
  held-out clients, nested L2           0.350    0.220    0.260       0.0782    1,790
  cause 2 (tuned on its own holdout): val P@50 0.240 -> 0.220 with nested L2 (chose 1.0)
  cause 1 (in-sample rows in the "full" score): 0.480 in-sample vs 0.240 held-out


B probe (+trend_pct_win)
  penalty grid scored on the val clients (what ML-08 optimised): {0.0001: 0.24, 0.001: 0.24, 0.01: 0.22, 0.1: 0.22, 1.0: 0.22} -> chose 0.0001
  slice                                  P@20     P@50    P@200         base     rows
  train rows (IN-SAMPLE)                0.600    0.520    0.455       0.1115   65,489
  held-out clients (reported)           0.300    0.240    0.265       0.0782    1,790
  full pool (mixed, ML-08's "full")     0.600    0.520    0.455       0.1107   67,279
  held-out clients, nested L2           0.350    0.220    0.260       0.0782    1,790
  cause 2 (tuned on its own holdout): val P@50 0.240 -> 0.220 with nested L2 (chose 1.0)
  cause 1 (in-sample rows in the "full" score): 0.520 in-sample vs 0.240 held-out

cross-check against work/outputs/ml08_metrics.json (same fit, must reproduce)
  A contract (5 feats)     shipped val P@50 0.24 vs here 0.240 | shipped full P@50 0.48 vs here 0.480
  B probe (+trend_pct_win) shipped val P@50 0.24 v


AFTER -- 6-fold client-grouped out-of-fold, penalty chosen inside each fold's train clients (A contract (5 feats))
  fold     hold clients     rows       pos      base         P@50
  1                   4      603        42    0.0697       0.2600
  2                   4   10,635       748    0.0703       0.3800
  3                   4    9,121     1,356    0.1487       0.5400
  4                   4   17,915     2,078    0.1160       0.3600
  5                   4    2,549       230    0.0902       0.4800
  6                   4   26,456     2,991    0.1131       0.3800
  pooled out-of-fold: P@20 0.350 | P@50 0.300 | P@200 0.385 | base 0.1107 | rows 67,279 | pos 7,445
  per-fold P@50 spread: min 0.260 | max 0.540 | sd 0.098  -> cause 3: one draw was carrying the claim
  penalty chosen per fold: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]



AFTER -- 6-fold client-grouped out-of-fold, penalty chosen inside each fold's train clients (B probe (+trend_pct_win))
  fold     hold clients     rows       pos      base         P@50
  1                   4      603        42    0.0697       0.2600
  2                   4   10,635       748    0.0703       0.3800
  3                   4    9,121     1,356    0.1487       0.5400
  4                   4   17,915     2,078    0.1160       0.3600
  5                   4    2,549       230    0.0902       0.4800
  6                   4   26,456     2,991    0.1131       0.3800
  pooled out-of-fold: P@20 0.250 | P@50 0.280 | P@200 0.375 | base 0.1107 | rows 67,279 | pos 7,445
  per-fold P@50 spread: min 0.260 | max 0.540 | sd 0.098  -> cause 3: one draw was carrying the claim
  penalty chosen per fold: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

BEFORE / AFTER, P@50 against the base rate (floor = 0.1107, ML-07 rule = 0.720)
  model                      BEFORE full pool (mixed) BEFORE val (L2 tuned on

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**DRAFT**

This is the Week-3 hunt pointed at the final feature set. I went after four families in order — a time window reaching past the decision point, the label reaching the feature matrix, a column that looks harmless and is not knowable yet, and a frame whose membership depends on the answer it is trying to predict. Three come back clean. The fourth is real, and measuring it changed my opinion about how much it matters.

**Time window: clean, and asserted rather than described.** Every feature carries a provenance row with its latest source date, and the cell asserts that no feature reads past the decision point and that every feature built from the daily performance table ends by 2026-02-28. The feature and label windows overlap by zero days — 2026-02-28 then 2026-03-01, adjacent, never shared. `content_age_days` is the one exemption and it earns it: it is computed *as of* the decision point from `content_created_date`, which is a static attribute knowable on the day. This is the structural property that answers Finding A's first question. My label is a forward outcome, not a re-measurement of the same fortnight.

**Label reach: clean.** Four label-side columns — `impr_label`, `label_rate_30d`, `drop_pct` and the deliberate leak I add later — are asserted absent from both feature sets, and the feature names are additionally grepped for month and label tokens so that a future edit quietly adding something like `impr_mar` fails loudly instead of passing review.

**The column that is not knowable yet: excluded, with the reason printed.** `dim_content.content_updated_date` is the tempting one — the freshest-looking signal in the dim table, and it would obviously correlate with a page about to lose impressions. It is unusable, and the cell shows why instead of asserting why: 76.2% of pool rows carry a `content_updated_date` after the decision point, with the latest at 2026-07-06. The column is an as-of-export snapshot, not a fact that existed on 2026-03-01. The honest response is to drop the column, not to trim it to the rows that happen to look early, because trimming would leave a feature whose availability depends on the export date.

**The fourth family, and where I was wrong about it.** The March inner join is a genuine design flaw. The frame outer-joins January and February, then inner-joins March, and the monthly aggregation keeps only days with positive impressions — so a page that stopped registering any positive-impression day in March has no March row to join and leaves the frame before the label is computed. Nobody is predicting from anything they should not know here; nothing crosses the boundary. This is outcome-conditioned population selection: who is in the study depends on what happened in the label window, and a page that vanished from search is the page most likely to be excluded.

I expected this to be the headline finding of the audit. Measured, it is not. It removes 235 pages out of 67,514 decided from January and February — 0.35% — and only 36 of those clear the 500-impression volume floor. Moving the base rate from 0.11066 to 0.11081, a shift of one and a half ten-thousandths. The reason is visible in the medians: the removed pages have a median prior rate of 173 impressions against 727 for the pages that stay, so they are small pages that were mostly going to fail the volume floor anyway. The mechanism is real and inherited from ML-04, and I would still flag it before anyone changes the floor or the join, because a lower floor would make that excluded group large and it is precisely the group that disappeared. But calling it something that caps every Precision@K above would have been wrong, and the number is the reason.

I also checked what the three defensible readings of the join actually do, because the scaffold picks one silently, and they do not agree. The inner join as built gives 0.11066. A left join with zeros filled gives 0.11081, and it labels the vanished pages correctly as declines. A left join with NaN kept gives 0.11027, the lowest of the three: those pages sit in the denominator but not the numerator, because `NaN <= -20` is False, so every vanished page reads as *not* a decline. That one is the dangerous reading, and it is dangerous precisely because it looks harmless. It does not crash, it does not produce a NaN base rate, and it leaves a plausible-looking precision figure sitting on a base rate that is 0.00039 too low. Every ranking number I could report would come out slightly flattering, with nothing in the output to tell me.

**And a trap, because an audit that finds nothing is indistinguishable from an audit that is broken.** The cell adds March impressions as a feature and re-runs the identical protocol. Precision@50 goes from 0.300 to 1.000, with five of six folds at a perfect 1.000. The evaluation reacts hard when information crosses the boundary, which is the evidence that the clean results above mean something.

In [3]:
# ================= (1) time window: does any feature read past the decision point? =================
for r in PROVENANCE:
    if r["role"] in ("feature", "probe feature"):
        assert pd.Timestamp(r["latest"]) <= pd.Timestamp(DECISION_POINT), r
    if r["role"] == "feature" and r["perf"]:
        assert pd.Timestamp(r["latest"]) <= pd.Timestamp(FEATURE_END), r
perf_feats = [r for r in PROVENANCE if r["role"] == "feature" and r["perf"]]
print("(1) window: %d contract features read the daily fact table, latest input %s, decision point %s"
      % (len(perf_feats), max(r["latest"] for r in perf_feats), DECISION_POINT))
print("    content_age_days is exempt on purpose: created_date is static and read as of the decision point,")
print("    latest input %s" % next(r["latest"] for r in PROVENANCE if r["col"] == "content_age_days"))
print("    feature and label windows overlap by %d days -> the label is a forward outcome, not a re-measurement"
      % max(0, (pd.Timestamp(FEATURE_END) - pd.Timestamp(LABEL_START)).days + 1))

# ================= (2) label reach: can the label get into the design matrix? =================
label_cols = [r["col"] for r in PROVENANCE if r["role"] in ("label only", "leak demo only")]
BANNED_TOKENS = ("mar", "apr", "may", "jun", "label", "drop", "future", "next")
for name, feats in MODELS:
    for fn in feats:
        assert fn not in label_cols, ("label column used as a feature", fn)
        low = fn.lower()
        assert not any(t in low for t in BANNED_TOKENS), ("feature name looks like a future column", fn)
    assert len(feats) == len(set(feats))
print()
print("(2) label reach: %d label-side columns (%s) asserted absent from both feature sets"
      % (len(label_cols), ", ".join(label_cols)))
print("    feature names also grepped for %s -> clean" % (", ".join(BANNED_TOKENS),))

# ================= (3) the column that is not knowable yet =================
upd = pd.to_datetime(P["content_updated_date"].astype("string"))
known = upd.notna()
share_post = float((upd[known] > pd.Timestamp(DECISION_POINT)).mean())
assert "content_updated_date" not in FEATS5 + FEATS6
print()
print("(3) content_updated_date: %.1f%% of pool rows with a non-null value carry a date AFTER %s"
      % (100*share_post, DECISION_POINT))
print("    %s pool rows, of which %s have a usable date; the column is an as-of-export snapshot,"
      % ("{:,}".format(len(P)), "{:,}".format(int(known.sum()))))
print("    not a fact that existed on the decision day, so it is excluded rather than trimmed.")
print("    example of a value we are refusing to use: %s" % upd[known].max().date())

# ================= (4) outcome-conditioned population selection (not a feature leak) =================
pre = j.merge(b, on=KEYS, how="outer")
pre["impr_prior"] = pre["impr_jan"].fillna(0) + pre["impr_feb"].fillna(0)
pre["prior_rate_30d"] = pre["impr_prior"]/PRIOR_DAYS*30
pre["in_pool"] = pre["impr_prior"] >= POOL_FLOOR
pool_pre = pre[pre["in_pool"]]
kept_keys = set(map(tuple, P[KEYS].to_numpy()))
dropped = pool_pre[[tuple(k) not in kept_keys for k in pool_pre[KEYS].to_numpy()]]
n_drop = int(len(dropped))
vol_ok = dropped["prior_rate_30d"] >= VOLUME_FLOOR
n_vol = int(vol_ok.sum())
base_left0 = (int(y.sum()) + n_vol) / (len(P) + n_drop)
# LEFT join with NaN kept: rows grow, positives do not, because NaN <= -20 is False
base_leftnan = int(y.sum()) / (len(P) + n_drop)
print()
print("(4) population selection: the decided pool vs the frame that survives the March inner join")
print("    pages decided from Jan+Feb alone (in_pool)      %10s" % "{:,}".format(len(pool_pre)))
print("    pages that survive inner-joining March          %10s" % "{:,}".format(len(P)))
print("    pages the March join removes                     %10s  (%.2f%% of the decided pool)"
      % ("{:,}".format(n_drop), 100.0*n_drop/len(pool_pre)))
print("    removed pages that clear the volume floor       %10s" % "{:,}".format(n_vol))
print("    reason: a page with no positive-impression day in March has no March row to join,")
print("    because agg_month keeps only days with gsc_impressions > 0.")
print("    for those pages March impressions are 0, so drop_pct is exactly -100 and the label is 1:")
print("    median prior_rate_30d of removed pages %.0f vs %.0f for kept pages"
      % (dropped["prior_rate_30d"].median(), P["prior_rate_30d"].median()))
print("    base rate on the frame as built                 %.5f  (%s pos / %s rows)"
      % (BASE, "{:,}".format(int(y.sum())), "{:,}".format(len(P))))
print("    base rate if the join were LEFT and 0 filled    %.5f  (%s pos / %s rows)"
      % (base_left0, "{:,}".format(int(y.sum())+n_vol), "{:,}".format(len(P)+n_drop)))
print("    base rate if the join were LEFT and NaN kept    %.5f  (%s pos / %s rows)"
      % (base_leftnan, "{:,}".format(int(y.sum())), "{:,}".format(len(P) + n_drop)))
print("      the %s removed rows are in the denominator and NOT in the numerator, because NaN <= -20 is"
      % "{:,}".format(n_drop))
print("      False. That drags the base rate DOWN by %.5f, which flatters every P@K above it."
      % (BASE - base_leftnan))
print("    -> three defensible readings, three different base rates, spanning %.5f. The scaffold picks"
      % (max(BASE, base_left0, base_leftnan) - min(BASE, base_left0, base_leftnan)))
print("       one silently, and the one it picks is not the middle one.")
print("    -> measured verdict: the mechanism is real, it is inherited from ML-04, and here it barely bites.")
print("       Under the reading that labels them correctly it removes %d rows (%.2f%%) and moves the base"
      % (n_drop, 100.0*n_drop/len(pool_pre)))
print("       rate by %+.5f; under the NaN reading it moves it by %+.5f. Either way the pages it removes are"
      % (base_left0 - BASE, base_leftnan - BASE))
print("       small pages that mostly fail the volume floor anyway. It is still worth naming, because the")
print("       pages it removes are the ones that stopped showing up, and a future floor change could")
print("       make that group large. It is NOT a feature leak: nothing crosses the boundary.")

# ================= (5) leak trap: prove the harness reacts to a real leak =================
leak_feats = tuple(FEATS5) + (LEAK,)
oof_leak, recs_leak, held_leak = run_oof(leak_feats)
res_leak = pak(oof_leak, y)
res_honest = AFTER["A contract (5 feats)"]["oof"]
print()
print("(5) leak trap: identical protocol, one extra feature = log1p(March impressions)")
print("    honest contract features  P@20 %.3f | P@50 %.3f | P@200 %.3f | base %.4f"
      % (res_honest["P@20"], res_honest["P@50"], res_honest["P@200"], res_honest["base"]))
print("    + the March label's own input P@20 %.3f | P@50 %.3f | P@200 %.3f | base %.4f"
      % (res_leak["P@20"], res_leak["P@50"], res_leak["P@200"], res_leak["base"]))
print("    lift %.3f -> %.3f = %.2fx, so the evaluation does react when information crosses the boundary."
      % (res_honest["P@50"], res_leak["P@50"], res_leak["P@50"]/res_honest["P@50"]))
print("    per-fold P@50 with the leak: %s" % [round(r["p50"], 3) for r in recs_leak])
print("    -> the contract features sit %.2fx above the random floor, the leaked model %.2fx above it."
      % (res_honest["P@50"]/BASE, res_leak["P@50"]/BASE))
LEAKRUN = dict(oof=res_leak, held=held_leak, feats=list(leak_feats))

(1) window: 4 contract features read the daily fact table, latest input 2026-02-28, decision point 2026-03-01


    content_age_days is exempt on purpose: created_date is static and read as of the decision point,
    latest input 2026-03-01
    feature and label windows overlap by 0 days -> the label is a forward outcome, not a re-measurement

(2) label reach: 4 label-side columns (log_impr_label_leak, impr_label, label_rate_30d, drop_pct) asserted absent from both feature sets
    feature names also grepped for mar, apr, may, jun, label, drop, future, next -> clean

(3) content_updated_date: 76.2% of pool rows with a non-null value carry a date AFTER 2026-03-01
    67,279 pool rows, of which 67,279 have a usable date; the column is an as-of-export snapshot,
    not a fact that existed on the decision day, so it is excluded rather than trimmed.
    example of a value we are refusing to use: 2026-07-06



(4) population selection: the decided pool vs the frame that survives the March inner join
    pages decided from Jan+Feb alone (in_pool)          67,514
    pages that survive inner-joining March              67,279
    pages the March join removes                            235  (0.35% of the decided pool)
    removed pages that clear the volume floor               36
    reason: a page with no positive-impression day in March has no March row to join,
    because agg_month keeps only days with gsc_impressions > 0.
    for those pages March impressions are 0, so drop_pct is exactly -100 and the label is 1:
    median prior_rate_30d of removed pages 173 vs 727 for kept pages
    base rate on the frame as built                 0.11066  (7,445 pos / 67,279 rows)
    base rate if the join were LEFT and 0 filled    0.11081  (7,481 pos / 67,514 rows)
    base rate if the join were LEFT and NaN kept    0.11027  (7,445 pos / 67,514 rows)
      the 235 removed rows are in the denominator and


(5) leak trap: identical protocol, one extra feature = log1p(March impressions)
    honest contract features  P@20 0.350 | P@50 0.300 | P@200 0.385 | base 0.1107
    + the March label's own input P@20 1.000 | P@50 1.000 | P@200 1.000 | base 0.1107
    lift 0.300 -> 1.000 = 3.33x, so the evaluation does react when information crosses the boundary.
    per-fold P@50 with the leak: [0.8, 1.0, 1.0, 1.0, 1.0, 1.0]
    -> the contract features sit 2.71x above the random floor, the leaked model 9.04x above it.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**DRAFT**

The boldest sentence I wrote in Week 5 was a reading of the coefficient table: that the model leans on prior volume, and that the baseline's edge comes from the Jan-to-Feb slope rather than from the five contracted features. Both halves were unsafe, and for the same reason — I was reading a fit's internals and calling them a fact about the world.

The problem is structural. A degree-2 expansion of five features has twenty-six coefficients, and they overlap: `log_impr_prior` appears linearly, squared, and multiplied by each of the other four, so its weight is spread across six columns that also carry everyone else's variance. Adding up absolute weights tells me how the fit distributed its penalty, which is a fact about the optimiser. So the cell measures importance the way I should have in Week 5 — permutation importance on held-out clients only, twenty shuffles per feature per fold, averaged over all six folds, model frozen and never refitted. A feature that matters stops predicting when its column is scrambled; decoration does not move the number.

One aggregation note before the numbers, because two Precision@50 values appear in this section and they are not the same quantity. Section 2's 0.300 is *pooled*: take all 67,279 held-out rows, sort once, read the top 50. The drops below are *per-fold*: each fold's held-out block is sorted on its own, so its top 50 comes from only four clients and the mean across folds is 0.400. Pooling weights a 26,456-row fold the same as a 603-row fold; per-fold does not. Both are legitimate, the comparison that matters here is a drop against its own 0.400 baseline, and I am labelling which is which rather than letting two different numbers share the name.

The two views disagree in a way that makes the point better than I could have argued. `active_days_prior` carries the **largest** share of the coefficient mass at 28.5%, and scrambling it costs **+0.0068** Precision@50, which at this sample size is indistinguishable from zero. `log_impr_prior` carries slightly less at 27.6% and scrambling it costs **0.2343**. And `ctr_prior_pct` lands at **-0.0180** — negative, meaning scrambling it helped slightly, which is noise rather than a weak signal. Coefficient mass is not importance; on this feature set the two rank almost oppositely.

The slope gets the sharpest correction. `trend_pct_win` is in the probe model because Week 5 believed it explained the baseline. It contributes 1.6% of the coefficient mass and scrambling it moves Precision@50 by **+0.0007**, and the probe model is *worse* out-of-fold than the contract model, 0.280 against 0.300. The honest statement is not "the slope matters a little". It is that I could not detect it at all, and that a feature I added on a story made the out-of-fold number worse.

So here is the rewrite, one clause at a time, with the measured number attached to each so the safe language is checkable rather than merely vaguer.

The claim I am replacing: *the model leans on prior volume, and the baseline wins because the slope matters.* Rewritten — **observed**: pooled across six client-grouped folds covering all 24 pool clients, the top 50 held 30.0% declines against an 11.1% base rate on the same rows, about 2.7x the random floor. **Measured**: scrambling prior volume costs 0.2343 Precision@50; the largest coefficient share costs 0.0068; the slope 0.0007. **Directional**: prior volume is the only feature in the set whose contribution I can detect at all, and only in the top 50 — which supports ordering a queue, not setting a threshold, not ranking clients, and not forecasting any date. **Decision-support**: use it to sequence a review queue for pages already carrying volume, and measure the queue against the 0.1107 floor before measuring it against anything else.

What I am withdrawing, explicitly: the slope sentence, and the word "wins" wherever I attached it to a comparison between 0.720 and 0.300. Those two numbers are on the same blended label but they are not measuring the same thing — ML-07's own forward February-to-March probe was 0.340 — so that comparison cannot settle which ranks March better, and I should not have implied it did.

What I cannot answer at all: whether any of this survives a second forward window. The lane is frozen at decision point 2026-03-01, I did not read April, and one forward period is not enough to know whether the ranking is stable. That is a limitation of the evidence, stated so nobody reads the 0.300 as a forecast.

In [4]:
# ================= importance measured on held-out clients, not read off the coefficients =========
def perm_importance(held, feats, k=50, reps=20, seed=11):
    """shuffle one raw feature column inside each held-out block, rebuild the design,
    rescore with the fold's own frozen model. A feature that matters loses P@K; decoration does not."""
    base_p50, drops = [], {f: [] for f in feats}
    for model, iva in held:
        D = mat(feats)[iva]
        yv = y[iva]
        p0 = prec_at_k(score(model, iva, feats), yv, k)
        base_p50.append(p0)
        for jj, fn in enumerate(feats):
            acc = []
            for r in range(reps):
                Dp = D.copy()
                col = D[:, jj].copy()
                np.random.default_rng(seed + 977*jj + 31*r).shuffle(col)
                Dp[:, jj] = col
                sp = build_design(Dp, model["med"], model["mu"], model["sd"], model["miss"]) @ model["w"]
                acc.append(p0 - prec_at_k(sp, yv, k))
            drops[fn].append(float(np.mean(acc)))
    return float(np.mean(base_p50)), {fn: float(np.mean(v)) for fn, v in drops.items()}

def coef_attention(model, feats):
    """share of |w| sitting on every term a feature appears in. This is what I read in Week 5,
    and it is a statement about how the fit spread its penalty, not about importance."""
    n = len(feats)
    names = ["intercept"] + ["lin:%s" % f for f in feats] + ["sq:%s" % f for f in feats]
    names += ["%s*%s" % (feats[i], feats[k]) for i in range(n) for k in range(i+1, n)]
    names += ["miss:%s" % f for f in feats]
    att = {f: 0.0 for f in feats}
    for nm, coef in zip(names, np.asarray(model["w"])):
        for f in feats:
            if f in nm:
                att[f] += abs(float(coef))
    tot = sum(att.values()) or 1.0
    return {f: 100.0*att[f]/tot for f in feats}

print("model A, out-of-fold P@50 %.4f against a base rate of %.4f" % (res_honest["P@50"], res_honest["base"]))
print()
print("%-22s %14s %18s" % ("feature", "|w| share %", "P@50 drop when shuffled"))
impA = {}
for name, _ in MODELS:
    feats = tuple(BEFORE[name]["feats"])
    held = AFTER[name]["held"]
    fold_p50, drops = perm_importance(held, feats)
    att = coef_attention(held[0][0], feats)
    impA[name] = dict(fold_p50=fold_p50, drops=drops, attention=att)
    print("--- %s | mean held-out P@50 in the folds used for attribution %.4f" % (name, fold_p50))
    print("%-22s %14s %18s" % ("", "", ""))
    for fn in sorted(feats, key=lambda x: -drops[x]):
        print("%-22s %13.1f%% %18.4f" % (fn, att[fn], drops[fn]))
    print("%-22s %13s %18.4f" % ("(base rate, none of this)", "-", BASE))
    print("    per-fold P@50 is computed inside each fold's own held-out block; it is not the pooled")
    print("    out-of-fold %.4f, because a fold of %s rows is a different top-50 than the whole pool."
          % (res_honest["P@50"], "{:,}".format(min(r["n"] for r in AFTER[name]["folds"]))))
    print()

# does the attribution method itself survive a check? point it at a model that really did leak
leak_p50, leak_drops = perm_importance(LEAKRUN["held"], tuple(LEAKRUN["feats"]))
print("sanity check on the deliberately leaked model (mean held-out P@50 %.3f):" % leak_p50)
for fn in sorted(LEAKRUN["feats"], key=lambda x: -leak_drops[x])[:3]:
    print("    %-22s P@50 drop when shuffled %+.4f" % (fn, leak_drops[fn]))
rank = [f for f in sorted(LEAKRUN["feats"], key=lambda x: -leak_drops[x])].index(LEAK)
print("    -> the injected leak ranks %d of %d here, not first: at P@50 %.3f the model is already"
      % (rank+1, len(LEAKRUN["feats"]), leak_p50))
print("       saturated, so scrambling any strong column costs almost as much. The check confirms the")
print("       method is not blind to the leak -- it is second-largest, at %+.4f -- but it does not cleanly"
      % leak_drops[LEAK])
print("       separate the leak from the volume column, and I am not going to pretend it does.")
print()

# ================= the rewrite, one clause at a time =================
oofA = res_honest
top = sorted(impA["A contract (5 feats)"]["drops"], key=lambda x: -impA["A contract (5 feats)"]["drops"][x])
print("=" * 92)
print("CLAIM LADDER -- every number below is printed by a cell above")
print("  UNSAFE (what I wrote in Week 5):")
print("    \"the model leans on prior volume, and the baseline wins because the slope matters\"")
print()
print("  OBSERVED       pooled across %d client-grouped folds covering all %d pool clients, the contract"
      % (K, int(P["client_hash_id"].nunique())))
print("                 model's top-50 queue held %.0f%% declines against a %.1f%% base rate on the same"
      % (100*oofA["P@50"], 100*oofA["base"]))
print("                 rows (%.1fx the random floor)." % (oofA["P@50"]/oofA["base"]))
dA = impA["A contract (5 feats)"]["drops"]
dB = impA["B probe (+trend_pct_win)"]["drops"]
aA = impA["A contract (5 feats)"]["attention"]
print("  MEASURED      scrambling %s inside the held-out blocks costs %.4f Precision@50."
      % (top[0], dA[top[0]]))
print("                 %s carries the LARGEST share of |w| (%.1f%%) and the smallest POSITIVE effect (%+.4f)."
      % (max(aA, key=aA.get), max(aA.values()), dA[max(aA, key=aA.get)]))
print("                 %s comes out NEGATIVE (%+.4f): scrambling it helped slightly, so it is noise."
      % (top[-1], dA[top[-1]]))
print("                 the Jan->Feb slope is only in the probe model, and it measures %+.4f there."
      % dB["trend_pct_win"])
print("  DIRECTIONAL   prior volume is the only feature in the set whose scrambling I can detect at all,")
print("                 and only in the top 50. Nothing here supports a threshold, a ranking order or a date.")
print("                 So I am withdrawing the Week-5 reading that the slope is what the baseline exploits:")
print("                 the probe model is %.3f out-of-fold against the contract model's %.3f, so adding the"
      % (AFTER["B probe (+trend_pct_win)"]["oof"]["P@50"], AFTER["A contract (5 feats)"]["oof"]["P@50"]))
print("                 slope made the honest number worse, not better.")
print("  DECISION-SUPPORT  use it to order a review queue for pages already carrying volume, and measure")
print("                 whether the queue beats the random floor before it beats anything else.")
print()
print("  WITHDRAWN     \"the model leans on prior volume\" is not wrong, it is unmeasured -- I had read")
print("                 it off coefficients. The measured version is above. The ML-07 comparison")
print("                 (%.3f rule vs %.3f out-of-fold model) is on the blended label and does not"
      % (M07["p_at_50"], oofA["P@50"]))
print("                 settle which ranks March better; ML-07's own forward probe was %.3f."
      % M07["forward_label_p_at_50"])
print("  UNANSWERABLE  whether any of this survives a second forward window. The lane stops at %s" % DECISION_POINT)
print("                 and I did not read April, so I have one forward period and not two.")

# ================= receipt =================
receipt = {
    "assignment": "ML-09",
    "audited_notebook": "work/notebooks/w05_model.ipynb",
    "paper": "docs/flyrank-seo-research-march-2026.pdf",
    "findings_audited": [
        {"id": "A", "title": "The Anatomy of Growing Content", "page": 6,
         "label": "trend_direction = 30d-vs-prev-30d impression change, same window as the profiled features",
         "my_answer": "my label window opens the day after my feature window closes (asserted)"},
        {"id": "B", "title": "What Predicts Growth?", "page": 29,
         "label": "same trend_direction buckets; split described as 80/20 on p.36, not grouped by brand",
         "my_answer": "base rate printed beside every P@K; 6-fold client-grouped out-of-fold evaluation"},
    ],
    "frame_rows": int(len(f)),
    "frame_clients": int(f["client_hash_id"].nunique()),
    "pool_rows": int(len(P)),
    "pool_clients": int(P["client_hash_id"].nunique()),
    "positives": int(y.sum()),
    "base_rate": BASE,
    "features_contract": FEATS5,
    "features_probe": FEATS6,
    "decision_point": DECISION_POINT,
    "seed": SEED,
    "oof_folds": K,
    "inner_folds_for_l2": N_INNER,
    "ml07_reference": {"p_at_50": M07["p_at_50"], "base_rate": M07["base_rate"],
                       "random_floor_p_at_50": M07["random_floor_p_at_50"],
                       "forward_p_at_50": M07["forward_label_p_at_50"]},
    "ml08_as_shipped": {"best_l2": M08["best_l2"], "model_A_val_p50": M08["model_A_val"]["P@50"],
                        "model_A_full_p50": M08["model_A_full"]["P@50"],
                        "model_B_val_p50": M08["model_B_val"]["P@50"],
                        "model_B_full_p50": M08["model_B_full"]["P@50"]},
    "before_after_p50": {n: {
        "before_full_pool_mixed_in_sample": BEFORE[n]["full_pool_mixed"]["P@50"],
        "before_val_l2_tuned_on_that_val": BEFORE[n]["val_reported"]["P@50"],
        "before_train_rows_in_sample": BEFORE[n]["train_in_sample"]["P@50"],
        "mid_val_nested_l2": BEFORE[n]["val_nested_l2"]["P@50"],
        "after_out_of_fold": AFTER[n]["oof"]["P@50"],
        "after_out_of_fold_p20": AFTER[n]["oof"]["P@20"],
        "after_out_of_fold_p200": AFTER[n]["oof"]["P@200"],
        "per_fold_p50": [r["p50"] for r in AFTER[n]["folds"]],
        "l2_shipped": BEFORE[n]["l2_shipped"], "l2_nested": BEFORE[n]["l2_nested"],
    } for n, _ in MODELS},
    "population_selection": {
        "decided_pool_rows": int(len(pool_pre)), "surviving_rows": int(len(P)),
        "removed_by_march_inner_join": n_drop,
        "removed_share_of_decided_pool": n_drop/len(pool_pre),
        "removed_clearing_volume_floor": n_vol,
        "base_rate_frame_as_built": BASE,
        "base_rate_if_left_join_zero_filled": base_left0,
     "base_rate_if_left_join_nan_kept": base_leftnan,
     "base_rate_spread_across_three_readings": max(BASE, base_left0, base_leftnan) - min(BASE, base_left0, base_leftnan),
        "verdict": "outcome-conditioned selection, inherited from ML-04; not a feature leak",
    },
    "leak_trap": {"feature_added": LEAK,
                  "honest_p50": res_honest["P@50"], "leaked_p50": res_leak["P@50"],
                  "lift": res_leak["P@50"]/res_honest["P@50"]},
    "permutation_importance_p50_drop": {n: impA[n]["drops"] for n, _ in MODELS},
    "coefficient_attention_share_pct": {n: impA[n]["attention"] for n, _ in MODELS},
    "leak_trap_importance_check": leak_drops,
    "withheld": ["no second forward window (April not read)",
                 "position sentence from paper finding A dropped rather than softened",
                 "ML-07 vs ML-09 comparison is on the blended label and does not settle March ranking"],
    "csv_committed": False,
}
out = REPO/"work"/"outputs"/"w06_validation_metrics.json"
out.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print()
print("receipt written:", out.relative_to(REPO).as_posix(), "(%d bytes)" % out.stat().st_size)

model A, out-of-fold P@50 0.3000 against a base rate of 0.1107

feature                   |w| share % P@50 drop when shuffled


--- A contract (5 feats) | mean held-out P@50 in the folds used for attribution 0.4000
                                                        
log_impr_prior                  27.6%             0.2343
wtd_pos_prior                   24.1%             0.0352
content_age_days                15.2%             0.0097
active_days_prior               28.5%             0.0068
ctr_prior_pct                    4.6%            -0.0180
(base rate, none of this)             -             0.1107
    per-fold P@50 is computed inside each fold's own held-out block; it is not the pooled
    out-of-fold 0.3000, because a fold of 603 rows is a different top-50 than the whole pool.



--- B probe (+trend_pct_win) | mean held-out P@50 in the folds used for attribution 0.4000
                                                        
log_impr_prior                  27.8%             0.2358
wtd_pos_prior                   23.2%             0.0407
content_age_days                14.8%             0.0153
active_days_prior               27.9%             0.0132
trend_pct_win                    1.6%             0.0007
ctr_prior_pct                    4.7%            -0.0132
(base rate, none of this)             -             0.1107
    per-fold P@50 is computed inside each fold's own held-out block; it is not the pooled
    out-of-fold 0.3000, because a fold of 603 rows is a different top-50 than the whole pool.



sanity check on the deliberately leaked model (mean held-out P@50 0.967):
    log_impr_prior         P@50 drop when shuffled +0.8812
    log_impr_label_leak    P@50 drop when shuffled +0.6090
    content_age_days       P@50 drop when shuffled +0.0013
    -> the injected leak ranks 2 of 6 here, not first: at P@50 0.967 the model is already
       saturated, so scrambling any strong column costs almost as much. The check confirms the
       method is not blind to the leak -- it is second-largest, at +0.6090 -- but it does not cleanly
       separate the leak from the volume column, and I am not going to pretend it does.

CLAIM LADDER -- every number below is printed by a cell above
  UNSAFE (what I wrote in Week 5):
    "the model leans on prior volume, and the baseline wins because the slope matters"

  OBSERVED       pooled across 6 client-grouped folds covering all 24 pool clients, the contract
                 model's top-50 queue held 30% declines against a 11.1% base rate on the sa

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Added for this assignment, left unticked until it is actually true:

- [ ] Section 1 quotes the paper by page and asks a question of my own work for each finding
- [ ] Section 2 shows the ML-08 split and a client-grouped out-of-fold number for both models, with base rate beside every P@K
- [ ] Section 3 asserts the window, asserts label reach, measures the outcome-conditioned frame selection, and demonstrates a real leak moving the number
- [ ] Section 4 replaces coefficient reading with permutation importance measured on held-out clients
- [ ] Metrics receipt written to `work/outputs/w06_validation_metrics.json`